<a href="https://colab.research.google.com/github/baudspan/MLPROJECT/blob/main/spam_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import time

In [ ]:
# Download the spam dataset
!wget https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv -O spam.tsv

--2026-03-09 20:15:36--  https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 477907 (467K) [text/plain]
Saving to: ‘spam.tsv’

spam.tsv            100%[===================>] 466.71K  --.-KB/s    in 0.009s  

2026-03-09 20:15:37 (52.0 MB/s) - ‘spam.tsv’ saved [477907/477907]



In [ ]:
data=list(range(100000))
start=time.time()
r=[x*2 for x in data]
t=time.time()-start
print("time taken:",t)

arr=np.array(data)
start=time.time()
rn=arr*2
tn=time.time()-start
print("time taken numpy:",tn)

print(f"NumPy was {t/tn:.1f}x faster")

time taken: 0.008885622024536133
time taken numpy: 0.0004963874816894531
NumPy was 17.9x faster


In [ ]:
df=pd.read_csv('spam.tsv',sep='\t',names=['label','message'])
message=df['message'].values

start=time.time()
r=[len(i) for i in message]
t=time.time()-start

start=time.time()
vectorized=np.vectorize(len)
rn=vectorized(message)
tn=time.time()-start

print("time taken:",t)
print("time taken numpy:",tn)
print(f"NumPy was {t/tn:.1f}x faster")
print("total message:",len(message))




time taken: 0.001104593276977539
time taken numpy: 0.0009002685546875
NumPy was 1.2x faster
total message: 5572


In [ ]:
df=pd.read_csv('spam.tsv',sep='\t',names=['label','message'])
print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
print("total message:",len(df))
print("\ndistribution")
print(df['label'].value_counts())
print("~~~~~~~~spam~~~~~~~~")
print(df[df['label']=='spam']['message'].iloc[0])
print("~~~~~~~~ham~~~~~~~~")
print(df[df['label']=='ham']['message'].iloc[0])


total message: 5572

distribution
label
ham     4825
spam     747
Name: count, dtype: int64
~~~~~~~~spam~~~~~~~~
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
~~~~~~~~ham~~~~~~~~
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


In [ ]:
df['label']=df['label'].map({'ham':0,'spam':1})
print("after conversion")
print(df.head())
print("label counts:",df['label'].value_counts())

after conversion
   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
label counts: label
0    4825
1     747
Name: count, dtype: int64


In [ ]:
x=df['message']
y=df['label']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
print("training message:",len(x_train))
print("testing message:",len(x_test))
print("training label(spam):",sum(y_train))
print("testing label(spam):",sum(y_test))

training message: 4457
testing message: 1115
training label(spam): 598
testing label(spam): 149


In [ ]:
vectorizer=TfidfVectorizer(max_features=3000,stop_words='english')
x_train_tfidf=vectorizer.fit_transform(x_train)
x_test_tfdif=vectorizer.transform(x_test)
print("training data shape:",x_train_tfidf.shape)
print("testing data shape:",x_test_tfdif.shape)

training data shape: (4457, 3000)
testing data shape: (1115, 3000)


In [ ]:
model=MultinomialNB()
model.fit(x_train_tfidf,y_train)
print("model trained!")
print("model type:",type(model))

model trained!
model type: <class 'sklearn.naive_bayes.MultinomialNB'>


In [ ]:
y_pred=model.predict(x_test_tfdif)
print("first 10 prediction")
print("predicted|actual")
print("-"*20)
for i in range(10):
  print(y_pred[i],'|',y_test.iloc[i])

first 10 prediction
predicted|actual
--------------------
0 | 0
0 | 0
0 | 0
0 | 0
0 | 0
0 | 0
0 | 0
0 | 0
0 | 0
0 | 0


In [ ]:
accuracy=accuracy_score(y_test,y_pred)
print("accuracy",accuracy*100,2)
print("confusion matrix")
cm=(confusion_matrix(y_test,y_pred))
print(cm)
print("\nReading the matrix:")
print("True Negatives (correctly identified ham):",(cm[0][0]))
print("False Positives (ham wrongly marked spam):",(cm[0][1]))
print("False Negatives (spam that got through):",(cm[1][0]))
print("True Positives (correctly caught spam):",(cm[1][1]))

print("\ndetailed report:")
print(classification_report(y_test,y_pred,target_names=['ham','spam']))

accuracy 98.47533632286995 2
confusion matrix
[[965   1]
 [ 16 133]]

Reading the matrix:
True Negatives (correctly identified ham): 965
False Positives (ham wrongly marked spam): 1
False Negatives (spam that got through): 16
True Positives (correctly caught spam): 133

detailed report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.99      0.89      0.94       149

    accuracy                           0.98      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
def test_message(message):
  message_tfidf=vectorizer.transform([message])
  prediction=model.predict(message_tfidf)[0]
  probablity=model.predict_proba(message_tfidf)[0]
  result="spam"if prediction==1 else"ham"
  spam_prob=probablity[1]*100
  print("message:",message)
  print("prediction:",result)
  print("spam probabity:",spam_prob,1)
  print("-"*50)

test_message("hey can we have a gmeet tomorrow at 4pm")
test_message("CONGRATULATIONS! You've WON $1000000 FREE! Click here NOW!!!")
test_message("Your Amazon order has been shipped")
test_message("FREE FREE FREE Click here to claim your prize!!!")

message: hey can we have a gmeet tomorrow at 4pm
prediction: ham
spam probabity: 1.6018018757476156 1
--------------------------------------------------
message: CONGRATULATIONS! You've WON $1000000 FREE! Click here NOW!!!
prediction: spam
spam probabity: 59.962208982634046 1
--------------------------------------------------
message: Your Amazon order has been shipped
prediction: ham
spam probabity: 23.880936031712384 1
--------------------------------------------------
message: FREE FREE FREE Click here to claim your prize!!!
prediction: spam
spam probabity: 87.51518418891733 1
--------------------------------------------------


In [ ]:
import pickle
pickle.dump(model,open('spam_model.pkl','wb'))
pickle.dump(vectorizer,open('vectorizer.pkl','wb'))
print("model and vectorizer saved")
from google.colab import files
files.download('spam_model.pkl')
files.download('vectorizer.pkl')

model and vectorizer saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>